In [1]:
%load_ext line_profiler

In [2]:
from ml_switching_reg_sim.data_creation import (
    MisclassificationCreator,
    UberDatasetCreator,
    UberDatasetCreatorHet,
)
from datetime import timedelta
import pandas as pd
import sys
import formulaic
import pyhdfe

sys.path.append("/Users/lordflaron/Documents/uganda-uber-paper/paper/scripts")
from regression import RegressionData
from ml_switching_reg.cm import cm_4, cm_10
from numba import jit

import autograd.numpy as np
import autograd as ag


In [3]:
# For simulations

# u = UberDatasetCreatorHet(drivers=1000, time_periods=30, regimes=3)

# df, mw, [beta0, beta1], y_sd = u.construct(
#     seed=1,
#     output_true_beta=True,
#     output_sigma=True,
#     y_sd=[1, 1, 1],
#     beta1=[-0.6, 0.2, 0.7],
#     beta0=[1, 2, 3],
#     weight=0.5,
# )

# beta_regime_0 = np.array([beta0[0], beta1[0]])[:, np.newaxis]
# beta_regime_1 = np.array([beta0[1], beta1[1]])[:, np.newaxis]
# beta_regime_2 = np.array([beta0[2], beta1[2]])[:, np.newaxis]

In [4]:
# For simulations
# R = df["regime"].values[:, np.newaxis]
# y = df["y"].values[:, np.newaxis]

# W0 = np.concatenate(
#     [np.ones((u.drivers * u.time_periods, 1)), df["drought_0"].values.reshape(-1, 1)],
#     axis=1,
# )  # add intercept
# W1 = np.concatenate(
#     [np.ones((u.drivers * u.time_periods, 1)), df["drought_1"].values.reshape(-1, 1)],
#     axis=1,
# )
# W2 = np.concatenate(
#     [np.ones((u.drivers * u.time_periods, 1)), df["drought_2"].values.reshape(-1, 1)],
#     axis=1,
# )

## IRLS

In [5]:
drop_date = "'2017-01'< date <='2019-12' and not '2018-01'<=date<='2018-07'"

lag_columns = [
    "intensity",
    "neg_intensity",
    "ind_low_intensity",
    "ind_intensity",
    "ind_pos_low_intensity",
    "ind_pos_intensity",
]

reg_data = RegressionData(
    outlier_quantile=0.99,
    drought_low=-1,
    drought_high=-1.5,
    lags=[
        1,
    ],
    lag_columns=lag_columns,
    drop_date=drop_date,
    driver_path="/Users/lordflaron/Documents/uganda-uber-paper/paper/data/uber_data/uganda_data/processed/drivers_all_hours.csv",
    weather_path="/Users/lordflaron/Documents/uganda-uber-paper/paper/data/weather/data/asap_data/raw/country_209_var_20_set_1_class_1_sensor_3.csv",
)

In [6]:
class MLSwitchingRegIRLS:
    
    def __init__(self, y, W, pi, R):
        self.y = y
        self.W = W
        self.pi = pi # rows are true labels and columns are predicted labels; same as sklearn
        self.R = R
        
        self.num_params = self.W[0].shape[1]  # change if different number of parameters per regime
        self.n_regimes = len(np.unique(self.R))
        
    # def yWb(self, beta):
        
    def Pr(self, sigma2, beta):
        
        W_mat = np.array(self.W)
        beta = beta.reshape(self.n_regimes, self.num_params, 1)
                
        regimes = np.exp(-((self.y.T - np.einsum("ijk,ikl -> ij", W_mat, beta)) ** 2)/2*sigma2)

        pi_r = np.einsum("ij,ik -> ijk", self.pi, regimes)
                
        F = self.pi.T @ regimes

        # cond_prob = self.pi[r, r_hat] * regimes[r] / F[r_hat] 
        # self.pi is with rows as true labels and columns as predicted labels
        # For ex: If the true regime is 0 and you're predicted to be regime 1, then you'll have
        # pi_01 * regime 0 as your true impact divided by the total probability of being chosen into regime 1

        cond_prob = pi_r / F
        
        return cond_prob
    
    def RR_hat(self, beta, sigma2):
        dummy_matrix = np.eye(self.n_regimes)[self.R].squeeze()
        RR = dummy_matrix.T * self.Pr(sigma2, beta)
        RR_hat = np.sum(RR, axis=1)
        return RR_hat
    
    def beta_h(self, r, rrh):
        return np.linalg.inv(
            np.linalg.multi_dot([self.W[r].T, np.diag(rrh[r].flatten()), self.W[r]])
        ).dot(np.linalg.multi_dot([self.W[r].T, np.diag(rrh[r].flatten()), self.y]))
        
    def sigma2_h(self, rrh, beta):
        W_mat = np.array(self.W)
        beta = beta.reshape(self.n_regimes, self.num_params, 1)
            
        sigma2_hat = np.sum(rrh * ((self.y.T - np.einsum("ijk,ikl -> ij", W_mat, beta)) ** 2))
        
        sigma2_hat = sigma2_hat / W_mat.shape[1]

        return sigma2_hat
    
    def jacobian(self, theta_old):
        rrh = self.RR_hat(theta_old[:-1], theta_old[-1])
                
        beta_hat = [self.beta_h(i, rrh) for i in range(self.n_regimes)]
        
        sigma2_hat = self.sigma2_h(rrh, theta_old[:-1])

        theta = np.vstack([
            np.array(beta_hat).flatten().reshape(self.num_params * self.n_regimes, 1),
            sigma2_hat,
        ])
        
        return theta - theta_old, theta
        
    def _irls(self, beta_0, sigma2_0,  
              max_iter=1000, tol=1e-6):
        
        theta_old = np.vstack([beta_0, sigma2_0])

        for i in range(max_iter):
            print(f"Starting iteration {i} with theta: {theta_old}")

            # calculate jacobian and get new theta
            jac, theta = self.jacobian(theta_old)
            
            norm = np.linalg.norm(jac)
                        
            print(f"Norm: {norm}")
            if norm < tol:
                theta_print = [f"{t:.3f}" for t in theta.flatten()]
                print(
                    f"Found in {i} iterations; with theta: {theta_print}, and norm: {norm}"
                )
                break
            theta_old = theta.copy()
        return theta
    
    def fit(self, beta_0, sigma2_0, max_iter=1000, tol=1e-6):
        return self._irls(beta_0, sigma2_0, max_iter, tol)

In [7]:
gaul_region_map_long_name = {
    "south highlands": "West",
    "west highlands": "West",
    "lake albert": "West",
    "eastern": "East",
    "west nile": "North",
    "mid northen": "North",
    "karamoja": "North",
    "lake victoria": "Central",
    "south drylands": "West",
    "south eastern": "East",
}


df_WMON = reg_data.regression_df(freq="W-MON")
df = reg_data.regression_df(freq="M")
df_2M = reg_data.regression_df(freq="2M")
df_3M = reg_data.regression_df(freq="BQ-JAN")

df_all_gaul = (
    reg_data.regression_df(freq="M", weather="all_weather")
    .reset_index()
    .drop_duplicates(["gaul", "date"])
    .groupby(["gaul", "date"])["lagged_1_neg_intensity"]
    .mean()
    .unstack(0)
)

df_all_gaul_2M = (
    reg_data.regression_df(freq="2M", weather="all_weather")
    .reset_index()
    .drop_duplicates(["gaul", "date"])
    .groupby(["gaul", "date"])["lagged_1_neg_intensity"]
    .mean()
    .unstack(0)
)

df_all_gaul_3M = (
    reg_data.regression_df(freq="BQ-JAN", weather="all_weather")
    .reset_index()
    .drop_duplicates(["gaul", "date"])
    .groupby(["gaul", "date"])["lagged_1_neg_intensity"]
    .mean()
    .unstack(0)
)

df_all_gaul_WMON = (
    reg_data.regression_df(freq="W-MON", weather="all_weather")
    .reset_index()
    .drop_duplicates(["gaul", "date"])
    .groupby(["gaul", "date"])["lagged_1_neg_intensity"]
    .mean()
    .unstack(0)
)

df_gaul = df.reset_index().merge(df_all_gaul, on="date")

df_gaul_2M = df_2M.reset_index().merge(df_all_gaul_2M, on="date")

df_gaul_WMON = df_WMON.reset_index().merge(df_all_gaul_WMON, on="date")

df_gaul_3M = df_3M.reset_index().merge(df_all_gaul_3M, on="date")

/Users/lordflaron/Documents/uganda-uber-paper/paper/scripts/regression.py:520: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda x: x.fillna(method='pad'))
/Users/lordflaron/Documents/uganda-uber-paper/paper/scripts/regression.py:520: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .appl

In [8]:
formula = lambda w: formulaic.Formula(
    f"np.arcsinh(hours_online) ~ -1 + `{w}` + shared_vehicle_1_0"
)


def irls_data(formula, df):
    fe = pyhdfe.create(
        df[["hashed_driver_uuid", "month", "year"]].values,
        drop_singletons=True,
    )

    singletons = fe.singleton_indices

    gauls = [
        "eastern",
        "karamoja",
        "lake albert",
        "lake victoria",
        "mid northen",
        "south eastern",
        "south drylands",
        "south highlands",
        "west nile",
        "west highlands",
    ]

    y = formula("eastern").get_model_matrix(df).lhs
    y_demeaned = fe.residualize(y)

    W = []
    W_demeaned = []

    for r in gauls:
        W.append(formula(r).get_model_matrix(df).rhs)
        W_demeaned.append(fe.residualize(formula(r).get_model_matrix(df).rhs))

    R = pd.Categorical(
        df.gaul_class,
        categories=gauls,
    ).codes[:, np.newaxis][~singletons]  # also drop the singletons

    x0 = np.ones((W[0].shape[1] * len(W), 1))
    
    mod = MLSwitchingRegIRLS(y = y_demeaned, 
                             W = W_demeaned, 
                             pi=cm_10,
                             R = R)

    return mod.fit(x0, 1)


In [9]:
# 3 month results

params_3M = irls_data(formula, df_gaul_3M)

Starting iteration 0 with theta: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]
Norm: 5.667140845632875
Starting iteration 1 with theta: [[ 0.29858166]
 [-0.5264081 ]
 [-0.4314985 ]
 [-0.80819085]
 [ 0.52080654]
 [-0.30531327]
 [ 0.31446038]
 [-0.30247955]
 [ 0.17436792]
 [-0.34917377]
 [ 0.26242858]
 [-0.23079152]
 [ 0.53841083]
 [-0.44984808]
 [ 0.97044227]
 [-0.64701865]
 [-0.93088965]
 [-0.56245673]
 [ 0.45929744]
 [-0.33829031]
 [ 2.42600681]]
Norm: 4.531063080475863
Starting iteration 2 with theta: [[ 0.31932942]
 [-0.93180893]
 [-3.30201384]
 [-2.56882928]
 [ 1.26138974]
 [ 0.02514335]
 [ 0.10416446]
 [-0.06731323]
 [ 0.06858431]
 [ 0.1564458 ]
 [ 0.03638584]
 [ 0.31263541]
 [ 1.08255277]
 [-0.72243701]
 [ 2.3849441 ]
 [-1.43748926]
 [-3.01299991]
 [-0.93695906]
 [ 0.24261019]
 [-0.07585917]
 [ 2.16468075]]


KeyboardInterrupt: 

In [60]:
# keep for monthly results
params_M = irls_data(formula, df_gaul)

Starting iteration 0 with theta: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]
Norm: 5.303439877254895
Starting iteration 1 with theta: [[ 0.53320063]
 [-0.60976891]
 [ 0.2754783 ]
 [-0.49881714]
 [ 0.89915378]
 [-0.4137825 ]
 [ 0.51506927]
 [-0.41164047]
 [ 0.25418463]
 [-0.07991526]
 [ 0.60186442]
 [-0.28644195]
 [ 0.45725739]
 [-0.38765899]
 [ 1.40470957]
 [-0.52780093]
 [-0.40049903]
 [-0.5672972 ]
 [ 0.77900718]
 [-0.45786849]
 [ 2.86089322]]
Norm: 4.382534371948075
Starting iteration 2 with theta: [[ 0.95729265]
 [-1.67158738]
 [-0.60436614]
 [-0.83502003]
 [ 2.61663705]
 [-0.32547334]
 [ 0.51926627]
 [-0.4647546 ]
 [-0.40012046]
 [ 1.58851641]
 [ 1.05805998]
 [ 0.44993454]
 [-0.03750238]
 [-0.06181816]
 [ 3.51153649]
 [-0.82956503]
 [-2.51989053]
 [-1.1977383 ]
 [ 1.23329827]
 [-0.61554252]
 [ 2.56365964]]


KeyboardInterrupt: 

In [28]:
## 2 month results
params_2M = irls_data(formula, df_gaul_2M)

Starting iteration 0 with theta: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]
Norm: 6.541560874952173
Starting iteration 1 with theta: [[-0.22706023]
 [-0.64081635]
 [-1.4124897 ]
 [-0.57346224]
 [-0.33221738]
 [-0.43837658]
 [ 0.37694118]
 [-0.39231612]
 [-0.56969075]
 [-0.18590372]
 [-0.51774892]
 [-0.37412261]
 [ 0.29295645]
 [-0.43683759]
 [-0.20440289]
 [-0.44731338]
 [-0.19664796]
 [-0.58185827]
 [ 0.24897388]
 [-0.39594767]
 [ 2.87322132]]
Norm: 1.9732272723252249
Starting iteration 2 with theta: [[-0.18949557]
 [-0.75908007]
 [ 0.344769  ]
 [-0.46074681]
 [ 0.04470149]
 [-0.38031239]
 [-0.04712677]
 [-0.41234116]
 [-0.24976191]
 [-0.21829287]
 [-0.35884421]
 [-0.36113345]
 [ 0.07687867]
 [-0.39526665]
 [ 0.27516084]
 [-0.28386261]
 [-0.2734218 ]
 [-0.65846775]
 [ 0.19769843]
 [-0.41340045]
 [ 2.93028629]]
Norm: 0.9845302119169481
Starting iteration 3 with theta: [[-0.23448082]
 [-0.76800492]
 [-0.

In [13]:
# weekly results

params_WMON = irls_data(formula, df_gaul_WMON)

Starting iteration 0 with theta: [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]


: 